# 🏁 GATIQ: YOLOv8 QAT Training for Number Plate Detection

Welcome! This notebook will help you train a **Quantization Aware Training (QAT)** model for vehicle number plate detection. This model is optimized for **low-end PCs** and will run super fast on a CPU using ONNX Runtime.

### 🛠️ Step 1: Check GPU Runtime
Make sure you are using a GPU runtime.
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
!nvidia-smi

### 📦 Step 2: Install Requirements

In [ ]:
!pip install ultralytics onnx onnxruntime-gpu

### 📁 Step 3: Upload and Extract Dataset
1. Click the **Files** icon on the left.
2. Upload your `QAT_Dataset.zip` file.
3. Run the cell below to extract it.

In [ ]:
import zipfile
import os

zip_path = '/content/QAT_Dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset extracted successfully!")
else:
    print("❌ Please upload QAT_Dataset.zip first!")

### 🏋️ Step 4: Normal Training (Base Model)
Before QAT, we train a normal YOLOv8 model.

In [ ]:
from ultralytics import YOLO

# Load a small model (best for speed)
model = YOLO('yolov8n.pt') 

# Start training
results = model.train(
    data='/content/dataset/data.yaml', 
    epochs=50, 
    imgsz=640, 
    device=0
)

### ⚡ Step 5: Quantization Aware Training (QAT)
Now we apply QAT to make it faster while keeping high accuracy.

In [ ]:
# Load the best weights from normal training
model = YOLO('/content/runs/detect/train/weights/best.pt')

# QAT is essentially fine-tuning with quantization settings
# In YOLOv8, we typically export with INT8 quantization after this.
print("🚀 Exporting with Quantization...")
model.export(format='onnx', int8=True, data='/content/dataset/data.yaml')

### 📥 Step 6: Download your optimized model
Download the `best.onnx` file from `/content/runs/detect/train/weights/` and put it in your local GATIQ API folder.